In [1]:
from pathlib import Path
import re
import pydicom
from pydicom.uid import ExplicitVRLittleEndian
import numpy as np
from PIL import Image
from pydicom.uid import JPEG2000Lossless

In [2]:
HD21_SCAN_WIDTH_MM = 9.0  # The width of the HD21 scan in millimeters, for pixel spacing metadata
INPUT_DIR = Path("../AutoExport_Raj/CZMI1260567209")
BASE_DICOM_PATH = Path("./zeiss_export/DATAFILES/E584/1BO5BKDBYO0B857TZSUQ3LBV9D4KKGEOM27IOLS4X2ZU.EX.DCM")
COMPRESS_OUTPUT = False
CLEAN_TEXT_ELEMENTS = True
USE_GENERIC_IOD = True

In [3]:
ds = pydicom.dcmread(BASE_DICOM_PATH)

In [4]:
# 1. Define your input directory with a broad glob
file_paths = list(INPUT_DIR.glob("**/*.tiff"))

hd21_files_grouped = {}
for filepath in file_paths:
    match = re.search(r'HD 21 Line_(\d+)_', filepath.name)
    if match:
        datetime_key = match.group(1)
        if not datetime_key in hd21_files_grouped:
            hd21_files_grouped[datetime_key] = []
        hd21_files_grouped[datetime_key].append(filepath)

# Sort numerically, by line #, rather than alphabetically
def extract_line_number(filepath):
    match = re.search(r'line(\d+)\.tif', filepath.name)
    if match:
        return int(match.group(1))
    return -1 # Fallback in case a file doesn't match perfectly
for datetime_key in hd21_files_grouped:
    hd21_files_grouped[datetime_key].sort(key=extract_line_number)
    assert len(hd21_files_grouped[datetime_key]) == 21, f"Expected 21 files for {datetime_key}, but found {len(hd21_files_grouped[datetime_key])}."

print(f"Number of HD 21 file groups: {len(hd21_files_grouped)}")

Number of HD 21 file groups: 2


In [5]:
file_group = list(hd21_files_grouped.values())[0] # use first group for demo
frames = []
for file in file_group:
    # Open, convert to 8-bit grayscale, and cast to uint8 array
    pixel_array = np.array(Image.open(file).convert('L'), dtype=np.uint8)
    frames.append(pixel_array)

# Merge the list of 2D arrays into a 3D array: shape becomes (21, Rows, Columns)
volume_array = np.stack(frames, dtype=np.uint8)

In [6]:
# Set basic frame and spatial dimensions
ds.NumberOfFrames = volume_array.shape[0]
ds.Rows = volume_array.shape[1]
ds.Columns = volume_array.shape[2]
# Infer pixel bit depth dynamically from numpy dtype
assert volume_array.dtype == np.uint8
ds.SamplesPerPixel = 1
ds.BitsAllocated = volume_array.itemsize * 8
ds.BitsStored = ds.BitsAllocated
ds.HighBit = ds.BitsAllocated - 1
ds.PixelRepresentation = 0  # 0 = Unsigned, 1 = Signed

ds.PhotometricInterpretation = "MONOCHROME2"

# Update Pixel Spacing
pixel_spacing = HD21_SCAN_WIDTH_MM / ds.Columns
ds.PixelSpacing = [pixel_spacing, pixel_spacing] # isotropic image

# Remove old Windowing tags so the viewer auto-windows the image
if "WindowCenter" in ds:
    del ds.WindowCenter
if "WindowWidth" in ds:
    del ds.WindowWidth

# Set endianness and VR explicit mode
ds.is_little_endian = True
ds.is_implicit_VR = False

if COMPRESS_OUTPUT:
    # Encodes pixels and automatically updates ds.PixelData & ds.file_meta.TransferSyntaxUID
    ds.compress(JPEG2000Lossless, volume_array)
else:
    # Explicit VR Little Endian Transfer Syntax UID for uncompressed raw data
    ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds.PixelData = volume_array.tobytes()

/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (46) exceeds the maximum length of 16 allowed for VR SH.
  warn_and_log(msg)


In [7]:
def clean_nulls(value):
    """Recursively clean strings, bytes, or lists of values."""
    if isinstance(value, str):
        # Truncate at the very first null character
        return value.split('\x00')[0]
    elif isinstance(value, bytes):
        # Truncate at the very first null byte
        return value.split(b'\x00')[0]
    elif isinstance(value, (list, pydicom.multival.MultiValue)):
        # Handle fields with multiple values (e.g., "Value1\Value2")
        return [clean_nulls(v) for v in value]
    return value

def clean_text_elements_callback(dataset, data_element):
    """Callback function for ds.walk() to target text VRs."""
    # List of all DICOM VRs that are represented as text/strings
    text_vrs = [
        'AE', 'AS', 'CS', 'DA', 'DT', 'LO', 'LT', 'PN', 
        'SH', 'ST', 'TM', 'UI', 'UR', 'UT', 'DS', 'IS'
    ]
    
    if data_element.VR in text_vrs and data_element.value is not None:
        # Clean the value
        cleaned_value = clean_nulls(data_element.value)
        # Reassigning the value tells pydicom to recalculate the length on save
        data_element.value = cleaned_value

if CLEAN_TEXT_ELEMENTS:
    ds.walk(clean_text_elements_callback)

/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (86) exceeds the maximum length of 64 allowed for VR UI. Invalid value for VR UI: '1.2.276.0.75.2.2.42.215507181855990.20211022105911826.5940159490\x00\x91Þz~\x02\x00\x00°\x91Þz~\x02\x00\x00p\x96Þz~\x02'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (118) exceeds the maximum length of 64 allowed for VR UI. Invalid value for VR UI: '1.2.276.0.75.2.2.42.215507181855990.20211022105737494.5940955490\x00M\x00z~\x02\x00\x00p\x91Þz~\x02\x00\x00\x90\x92Þz~\x02\x00\x00°\x8fÞz~\x02\x00\x00\x90\x8fÞz~\x02\x00\x00°\x95Þz~\x02\x00\x00p\x90Þz~\x02'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for

In [8]:
if USE_GENERIC_IOD:
    # "Multi-frame Grayscale Byte Secondary Capture Image Storage"
    # Generic multi-frame 8-bit grayscale image
    generic_multiframe_uid = "1.2.840.10008.5.1.4.1.1.7.2"

    # 2. Update main dataset & file meta header
    ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.7.2"
    ds.file_meta.MediaStorageSOPClassUID = generic_multiframe_uid

# Generate new SOP Instance UID
ds.SOPInstanceUID = pydicom.uid.generate_uid()
ds.file_meta.MediaStorageSOPInstanceUID = ds.SOPInstanceUID

/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: '1.2.840.10008.5.1.4.1.1.12.77\x00\x00\x00\x00\x00\x00\x00>\x00\x00\x00\x08\x0b´|~\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00±ª'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (102) exceeds the maximum length of 64 allowed for VR UI. Invalid value for VR UI: '1.2.276.0.75.2.2.42.215507181855990.20211022105911826.5940159490\x00\rô«~\x02\x00\x00\x00\x0eô«~\x02\x00\x00@\rô«~\x02\x00\x00À\x0bô«~\x02\x00\x00À\x0eô«~\x02'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


In [9]:
output_filepath = Path(f"hd21{'_compressed' if COMPRESS_OUTPUT else ''}{'_cleaned' if CLEAN_TEXT_ELEMENTS else ''}{'_generic' if USE_GENERIC_IOD else ''}.dcm")
ds.save_as(output_filepath)
print(f"DICOM file successfully overwritten with {ds.NumberOfFrames} frames!")

DICOM file successfully overwritten with 21 frames!


In [10]:
# Read the output dicom to ensure validity
ds = pydicom.dcmread(output_filepath)